In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"] = "4"

import torch
torch.set_num_threads(4)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/bpic20_Dom.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "org:resource": "string",
        "org:role": "string",
        "case:Amount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:Amount,concept:name,org:resource,org:role,time_delta
0,declaration 100000,2018-01-30 09:20:07,600.844116,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
1,declaration 100000,2018-02-07 09:58:46,600.844116,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,693519.0
2,declaration 100000,2018-02-08 10:59:05,600.844116,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,90019.0
3,declaration 100000,2018-02-09 12:42:49,600.844116,Request Payment,SYSTEM,UNDEFINED,92624.0
4,declaration 100000,2018-02-12 17:31:20,600.844116,Payment Handled,SYSTEM,UNDEFINED,276511.0
5,declaration 100005,2018-01-30 09:38:54,35.133686,Declaration SUBMITTED by EMPLOYEE,STAFF MEMBER,EMPLOYEE,0.0
6,declaration 100005,2018-01-30 09:38:57,35.133686,Declaration APPROVED by ADMINISTRATION,STAFF MEMBER,ADMINISTRATION,3.0
7,declaration 100005,2018-01-30 10:04:10,35.133686,Declaration FINAL_APPROVED by SUPERVISOR,STAFF MEMBER,SUPERVISOR,1513.0
8,declaration 100005,2018-01-31 12:45:18,35.133686,Request Payment,SYSTEM,UNDEFINED,96068.0
9,declaration 100005,2018-02-01 17:31:17,35.133686,Payment Handled,SYSTEM,UNDEFINED,103559.0


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:Amount', 'concept:name', 'org:resource', 'org:role', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [2.00, 284911.00]                        42289.5000 quantile_derived    
case:Amount                    continuous     case     yes    [6.91, 219.03]                           25.3885    quantile_derived    
org:resource                   categorical    event    yes    ['STAFF MEMBER', 'SYSTEM']               N/A        data_derived        
org:role                       categorical    event    ye

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[]

In [14]:
engine.branching_sets

[{'Declaration APPROVED by ADMINISTRATION',
  'Declaration APPROVED by BUDGET OWNER',
  'Declaration APPROVED by PRE_APPROVER',
  'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration REJECTED by MISSING',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'},
 {'Declaration FOR_APPROVAL by SUPERVISOR',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by MISSING'},
 {'Declaration REJECTED by ADMINISTRATION',
  'Declaration REJECTED by PRE_APPROVER',
  'Declaration REJECTED by SUPERVISOR'}]

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic20_Dom-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/220 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,declaration 142992,4,1,0,0.328107,0.343715,0.312500,0.371429,0.272727,...,0.639149,0.272727,0.000000,0.0000,0.000000,0.00,0.639149,0.639149,0.999998,0.999998
1,0,declaration 115669,4,1,0,0.286625,0.260751,0.312500,0.350000,0.272727,...,0.674561,0.272727,0.000000,0.0000,0.000000,0.00,0.674561,0.674561,0.999999,0.999999
2,0,declaration 138710,4,1,0,0.360786,0.359073,0.362500,0.428571,0.268182,...,0.625557,0.272727,0.000000,0.0000,0.000000,0.00,0.625557,0.625557,0.999998,0.999998
3,0,declaration 141310,4,1,0,0.298378,0.334255,0.262500,0.335714,0.263636,...,0.674496,0.272727,0.000000,0.0000,0.000000,0.00,0.674496,0.674496,0.999999,0.999999
4,0,declaration 113587,5,1,0,0.328179,0.431358,0.225000,0.364286,0.234615,...,0.572704,0.230769,0.000000,0.0000,0.000000,0.00,0.572704,0.572704,0.999996,0.999996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,20,declaration 124561,11,1,0,0.376933,0.360115,0.393750,0.476000,0.400000,...,0.123929,0.400000,0.043929,0.0625,0.025358,0.08,0.000000,0.562970,0.000000,0.999995
148,20,declaration 134394,11,1,0,0.384355,0.431211,0.337500,0.428000,0.388000,...,0.142500,0.240000,0.062500,0.1250,0.000000,0.08,0.000000,0.000000,0.000000,0.000000
149,20,declaration 129484,11,1,0,0.318652,0.290429,0.346875,0.420000,0.400000,...,0.000000,0.400000,0.000000,0.0000,0.000000,0.00,0.000000,0.340776,0.000000,0.000000
150,20,declaration 126499,11,1,0,0.414142,0.415785,0.412500,0.514000,0.396000,...,0.132154,0.320000,0.052154,0.0625,0.041808,0.08,0.000000,0.308987,0.000000,0.000000


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()